## Process the customers data
###### 1. Ingest the data into the data lakehouse - bronze_customers
###### 2. Perform data quality checks and transform the data as required - siver_customers_clean
###### 3. apply changes to the Customers data - silver_customers

In [0]:
%fs ls /Volumes/circuitbox_sivan/landing/operational_data/customers/

### 1. Ingest the data into the data lakehouse - bronze_customers

###### Add columns
###### 1. Input_file_path
###### 2. Ingestion_timestamp

In [0]:
create or refresh streaming table bronze_customers
comment 'raw customers data ingested from the source system operational data'
tblproperties('quality' = 'bronze')
as
select *,
    _metadata.file_path as file_path,
    current_timestamp as ingestion_timestamp
from cloud_files(
    '/Volumes/circuitbox_sivan/landing/operational_data/customers/',
    'json',
    map('cloudFiles.inferColumnTypes', 'true')    
)

In [0]:
select * from circuitbox_sivan.lakehouse.bronze_customers;

#### 2. Perform data quality chekcks and transform the data as required - silver_customers_clean

Siver Customer clean (DQ Rules) [Data Quality Rules]
1. Fail if customer_id is NULL
2. Drop records with customer_name as Null
3. Warn if telephone is less than 10 characters
4. Warn if email is NULL
5. Warn if date_of_birth is befor 1920

Silver Customer Clean (Transofrmations)
1. CAST date_of_birth to DATE
2. CAST created_date to DATE


######## Source --> For each Reocrd --> Expectation --> true --> Inclued Repecord --> Target
########                            --> Expectation --> False--> Action --> Warn --> Inclued Repecord --> Target
########                                                     --> Action --> Fail --> Fail Data Flow
########                                                     --> Action --> Drop --> Drop Record

CONSTRAINT <expectation_name>
    EXPECT (Conditonal_expresssion>)
    [on Violation (Fail UPDATE /\ Drop Row)]    
